<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/4.State_Estimator_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validating Thermal ZOne Model in Energy Plus


### Setup Environement

In [ ]:
!pip uninstall -y energy-plus-utility

Found existing installation: energy-plus-utility 0.2.2+5
Uninstalling energy-plus-utility-0.2.2+5:
  Successfully uninstalled energy-plus-utility-0.2.2+5


In [ ]:
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5


In [ ]:
from eplus import prepare_colab_eplus
prepare_colab_eplus()

In [ ]:
!pip install control

## Setup Model

In [ ]:
from eplus.core import EPlusUtil
import types
import datetime

import pandas as pd
import numpy as np
import control as ct
import traceback


import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# @title Setup Model
OUT_DIR = "/simulation/eplus_out"
url_idf="https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

# Initialize Utility

sim = EPlusUtil(verbose=1, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)
sim.run_dry_run(include_ems_edd=False,reset=True,design_day=True)

# catalog = sim.api_catalog_df()
# mo.ui.table(catalog)
# mo.ui.table(catalog['VARIABLES'])
# sim.list_available_variables()

EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneAirCooled.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
EnergyPlus state has been reset.
EnergyPlus state has been reset.


0

## Setup Simulator

In [ ]:
# @title Setup Data Logger
# Request the variables to construct State Vector (x_i) and Disturbances (d_i)
specs = [
    # Zone States (x_i)
    {"name": "Zone Mean Air Temperature", "key": "*"},       # T_in,i
    {"name": "Zone Mean Radiant Temperature", "key": "*"},   # T_m,i (Thermal Mass proxy)
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},    # W_in,i
    {"name": "Zone Air Relative Humidity", "key": "*"},      # W_in,i (%)
    {"name": "Zone Air CO2 Concentration", "key": "*"},      # ppm

    # Time-Varying Parameters (p_i)
    {"name": "Zone People Occupant Count", "key": "*"},      # No. of People
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*"}, # Watts

    # External Environment Conditions (x_out)
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"}, # T_out
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},      # W_out
    {"name": "Site Outdoor Air Relative Humidity", "key": "*"},   # W_in,i (%)
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},     # ppm

    # Control Inputs - VAV Box Volumetric Flow Rate (m3/s)
    {"name": "System Node Current Density Volume Flow Rate", "key": "*"},

    # AHU Supply Parameters (S)
    {"name": "System Node Temperature", "key": "*"},       # T_s
    {"name": "System Node Humidity Ratio", "key": "*"},    # W_s
    {"name": "System Node CO2 Concentration", "key": "*"}, # C_s
]
sim.ensure_output_variables(specs, activate=True)

sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Extracts sensor data, updates the current snapshot, and logs history."""
    if not self.exchange.api_data_fully_ready(state):
        return

    # 1. Get Simulation Time Details
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # 2. Extract Outdoor and Supply Data
    t_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment")
    w_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Humidity Ratio", "Environment")
    rh_out_h = self.exchange.get_variable_handle(state, "Site Outdoor Air Relative Humidity", "Environment")
    co2_out_h = self.exchange.get_variable_handle(state, "Schedule Value", "CO2-Outdoor-Actuated")

    row["T_out"] = self.exchange.get_variable_value(state, t_out_h)
    row["W_out"] = self.exchange.get_variable_value(state, w_out_h)
    row["RH_out_%"] = self.exchange.get_variable_value(state, rh_out_h)
    row["CO2_out"] = self.exchange.get_variable_value(state, co2_out_h)


    t_s_h = self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node")
    w_s_h = self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    c_s_h = self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node")

    row["T_s"] = self.exchange.get_variable_value(state, t_s_h)
    row["W_s"] = self.exchange.get_variable_value(state, w_s_h)
    row["C_s"] = self.exchange.get_variable_value(state, c_s_h)

    # 3. Extract Internal States for All Zones
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for zone in zones:
        t_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone)
        t_m_h = self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone)
        w_in_h = self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone)
        rh_in_h = self.exchange.get_variable_handle(state, "Zone Air Relative Humidity", zone)
        co2_in_h = self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone)
        occ_in_h = self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone)
        q_eq_h = self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone)
        v_dot_h = self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone} In Node")

        row[f"{zone}_T_in"] = self.exchange.get_variable_value(state, t_in_h)
        row[f"{zone}_T_m"] = self.exchange.get_variable_value(state, t_m_h)
        row[f"{zone}_W_in"] = self.exchange.get_variable_value(state, w_in_h)
        row[f"{zone}_RH_%"] = self.exchange.get_variable_value(state, rh_in_h)
        row[f"{zone}_CO2_in"] = self.exchange.get_variable_value(state, co2_in_h)
        row[f"{zone}_Occ"] = self.exchange.get_variable_value(state, occ_in_h)
        row[f"{zone}_Q_equip"] = self.exchange.get_variable_value(state, q_eq_h)
        row[f"{zone}_V_dot"] = self.exchange.get_variable_value(state, v_dot_h)

    # 4. Update the current snapshot AND append to historical log
    self.current_state = row
    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)
sim.register_handlers(
    "begin", [
        {"method_name": "state_logger"},
        {"method_name": "occupancy_handler", "kwargs": {"lam": 3.0, "min": 0, "max": 5, "seed": 4 } },
        {"method_name": "co2_set_outdoor_ppm", "kwargs": { "value_ppm": 420.0, "log_every_minutes": 60 } }
    ]
)

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

EnergyPlus state has been reset.
Handlers on 'begin' hook: ['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm']


In [ ]:
# @title zone_model
def zone_model(self, state):
    try:
        if not self.exchange.api_data_fully_ready(state):
            return
        zone_id = "SPACE5-1"

        # Time Stamping
        day = self.exchange.day_of_year(state)
        time = self.exchange.current_time(state)
        base_date = datetime.datetime(2026, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time * 3600)))
        abs_time = (day * 24.0) + time

        print("Zone :"+zone_id+" Time :"+str(base_date.strftime("%Y-%m-%d %H:%M:%S")))

        # Check If Zones is initialized
        if not hasattr(self, 'zones'):
            self.zones = {}

        # --- Initiallize Zone ---
        if zone_id not in self.zones:

            # --- Fetch Parameters and Initialize ---
            raw_params = self.get_zone_thermal_parameters()[zone_id]
            handles = {
                "T_in": self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", zone_id),
                "T_m": self.exchange.get_variable_handle(state, "Zone Mean Radiant Temperature", zone_id),
                "W_in": self.exchange.get_variable_handle(state, "Zone Mean Air Humidity Ratio", zone_id),
                "CO2_in": self.exchange.get_variable_handle(state, "Zone Air CO2 Concentration", zone_id),
                "N_occ": self.exchange.get_variable_handle(state, "Zone People Occupant Count", zone_id),
                "T_out": self.exchange.get_variable_handle(state, "Site Outdoor Air Drybulb Temperature", "Environment"),
                "Q_equip": self.exchange.get_variable_handle(state, "Zone Electric Equipment Total Heating Rate", zone_id),
                "V_dot": self.exchange.get_variable_handle(state, "System Node Current Density Volume Flow Rate", f"{zone_id} In Node"),
                "T_s": self.exchange.get_variable_handle(state, "System Node Temperature", "VAV Sys 1 Outlet Node"),
                "W_s": self.exchange.get_variable_handle(state, "System Node Humidity Ratio", "VAV Sys 1 Outlet Node"),
                "C_s": self.exchange.get_variable_handle(state, "System Node CO2 Concentration", "VAV Sys 1 Outlet Node"),
            }

            inv_R_env_ext = 0.0
            R_env_gnd = None
            adj_zones = []

            for b in raw_params["boundaries"]:
              target = b["target"]
              r_abs = float(b["R_absolute_K_W"])
              if target == "Ground": R_env_gnd = r_abs
              elif target == "Environment" or b["boundary_condition"] == "outdoors": inv_R_env_ext += (1.0 / r_abs)
              else: adj_zones.append({ "zone": target, "R_env": r_abs, "handle_T_in": self.exchange.get_variable_handle( state, "Zone Mean Air Temperature", target )})
            R_env_ext = 1.0 / inv_R_env_ext if inv_R_env_ext > 0 else float('inf')


            # --- Define Dynamics ---
            def _dynamics(t, x, u, params):
                T_in, T_m, W_in, C_in = x # Extract State Variables
                V_dot_s = float(u[0]) # Extract Control Variable

                # consts
                rho_air, cp_air = 1.204, 1006.0
                q_person, g_w_person, g_co2_person = 100.0, 5e-5, 1e-5
                R_env_ext = float(params.get('R_env_ext', float('inf')))
                R_env_gnd = float(params.get('R_env_gnd', float('inf')))
                R_int = float(params['R_int'])
                C_air = float(params['C_air'])
                C_mass = float(params['C_mass'])
                M_air = float(params['M_air'])
                V_room = float(params['V_room'])


                T_s = params['T_s'] # Supply Variables
                W_s = params['W_s']
                C_s = params['C_s']

                N_occ = params['N_occ'] # Time-Varying Parameters
                Q_equip = params['Q_equip']

                T_out = params['T_out'] # External Temperature

                d_T, d_W, d_C = params['d_T'], params['d_W'], params['d_C'] # Disturbances


                # ----- Temperature Dynamics
                q_env = (T_out - T_in) / R_env_ext if R_env_ext < float('inf') else 0.0
                q_gnd = (22.0 - T_in) / R_env_gnd if R_env_gnd < float('inf') else 0.0
                q_adj = 0.0
                for adj in params['adj_zones']:
                    q_adj += (float(adj['T_in']) - float(T_in)) / float(adj['R_env'])
                q_mass = (T_m - T_in) / R_int
                q_int = (N_occ * q_person) + Q_equip
                q_s = rho_air * V_dot_s * cp_air * (T_s - T_in)
                dT_in_dt = (q_env + q_gnd + q_adj + q_mass + q_int + q_s + d_T) / C_air

                # ----- Mass Temperature Dynamics
                dT_m_dt = (T_in - T_m) / ( C_mass * R_int)

                # ----- Humidity Dynamics
                dot_m_s = rho_air * V_dot_s
                dW_in_dt = (N_occ * g_w_person + dot_m_s * (W_s - W_in) + d_W) / M_air

                # ----- CO2 Dynamics
                dC_in_dt = (N_occ * g_co2_person + V_dot_s * (C_s - C_in) + d_C) / V_room

                return np.array([dT_in_dt, dT_m_dt, dW_in_dt, dC_in_dt], dtype=float).flatten()

            def _outputs(t, x, u, params):
                # return [x[0], x[2], x[3]] # Observable: T_in, W_in, C_in
                return [x[0], x[1], x[2], x[3]]

            sys_ode = ct.NonlinearIOSystem(
                _dynamics, _outputs,
                inputs=['V_dot_s'],
                outputs=['T_in_obs','T_m_obs', 'W_in_obs', 'C_in_obs'],
                states=['T_in', 'T_m', 'W_in', 'C_in'],
                name=f'sys_{zone_id}'
            )

            # --- Store in Namespace ---
            self.zones[zone_id] = types.SimpleNamespace(
                last_time=abs_time,
                V_room=float(raw_params['V_room']),
                M_air=float(raw_params['M_air']),
                C_air=float(raw_params['C_air']),
                C_mass=float(raw_params['C_mass']),
                R_int=float(raw_params['R_int']),
                R_env_gnd=R_env_gnd,
                R_env_ext=R_env_ext,
                adj_zones=adj_zones,
                handles=handles,
                sys_ode=sys_ode,
                log=[]
            )

            # --- Testing Print ---
            z = self.zones[zone_id]
            print(f"\n--- [{zone_id}] Param Extraction ---")
            print(f"Physical: V={z.V_room}, M_air={z.M_air}")
            print(f"R_ground: {z.R_env_gnd}")
            print(f"R_env (External Merged): {z.R_env_ext:.6f}")
            print(f"Adjacent Zones Array: {z.adj_zones}")
            print("-----------------------------------\n")
            print(f"[{zone_id}] Handles initialized. Neighbors: {[az['zone'] for az in adj_zones]}")
            print(sys_ode)

        else:
            self.zones[zone_id].last_time = abs_time

        z = self.zones[zone_id]
        # --- Run Dynamic System
        row = {
            "timestamp": base_date.strftime("%Y-%m-%d %H:%M:%S"),
        }

        u_current = [ self.exchange.get_variable_value(state, z.handles["V_dot"]) ]

        current_adj_zones = []
        for adj in z.adj_zones:
            current_adj_zones.append({
                'T_in': self.exchange.get_variable_value(state, adj["handle_T_in"]),
                'R_env': adj["R_env"]
            })

        current_params = {
            'C_air': z.C_air, 'C_mass': z.C_mass,
            'R_env_ext': z.R_env_ext, 'R_env_gnd':z.R_env_gnd,
            'R_int': z.R_int, 'M_air': z.M_air, 'V_room': z.V_room,
            'T_out': self.exchange.get_variable_value(state, z.handles["T_out"]), # Dynamically updated disturbances from E+
            'N_occ': self.exchange.get_variable_value(state, z.handles["N_occ"]),
            'Q_equip': self.exchange.get_variable_value(state, z.handles["Q_equip"]),
            'T_s': self.exchange.get_variable_value(state, z.handles["T_s"]),# Supply Data
            'W_s': self.exchange.get_variable_value(state, z.handles["W_s"]),
            'C_s': self.exchange.get_variable_value(state, z.handles["C_s"]),
            'd_T': 0.0, 'd_W': 0.0, 'd_C': 0.0,# Observer variables (update these from your EKF later)
            'adj_zones': current_adj_zones # Adjacent Zones
            }

        has_pred = hasattr(z, 'prev_prediction')

        t_in_true = self.exchange.get_variable_value(state, z.handles["T_in"])
        t_m_true  = self.exchange.get_variable_value(state, z.handles["T_m"])
        w_in_true = self.exchange.get_variable_value(state, z.handles["W_in"])
        c_in_true = self.exchange.get_variable_value(state, z.handles["CO2_in"])
        x_actual = [t_in_true, t_m_true, w_in_true, c_in_true]
        has_pred = hasattr(z, 'prev_prediction')
        x_solver = [
            t_in_true,  # Measurable
            # z.prev_prediction[1] if has_pred else t_in_true, # Hidden (Use prediction!)
            t_m_true,   # Measurable
            w_in_true,  # Measurable
            c_in_true   # Measurable
        ]

        dt_hours = self.exchange.system_time_step(state)
        if dt_hours == 0: # Fallback to zone step if system step isn't active yet
            dt_hours = self.exchange.zone_time_step(state)
        time_vector = [0, dt_hours * 3600.0]

        response = ct.input_output_response(z.sys_ode, time_vector, U=u_current, X0=x_solver, params=current_params)
        x_predicted_next = response.states[:, -1]

        row = {
            "timestamp": base_date,
            "T_in_actual": x_actual[0],
            "T_in_pred": z.prev_prediction[0] if has_pred else 0,
            "T_m_actual": x_actual[1],
            "T_m_pred": z.prev_prediction[1] if has_pred else 0,
            "W_in_actual": x_actual[2],
            "W_in_pred": z.prev_prediction[2] if has_pred else 0,
            "C_in_actual": x_actual[3],
            "C_in_pred": z.prev_prediction[3] if has_pred else 0,
            "T_out": current_params['T_out'],
            "V_dot_s": u_current[0]
        }
        z.log.append(row)
        z.prev_prediction = x_predicted_next

    except Exception as e:
        print(f"\n--- Python Exception in zone_1_model ---")
        print(f"Error: {e}")
        traceback.print_exc()
        print("----------------------------------------\n")

sim.zone_model = types.MethodType(zone_model, sim)
sim.register_handlers("begin", [{"method_name": "zone_model"}])

['state_logger', 'occupancy_handler', 'co2_set_outdoor_ppm', 'zone_model']

In [ ]:
# @title Run Simulation
# Set Simulation Time
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 31),
    timestep_per_hour = 6, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Sunday",
)

print("Starting EnergyPlus Uncontrolled Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete! Converting data to Pandas...")
    # Create the DataFrame
    SimulationData = pd.DataFrame(sim.collected_data)
    cols = ['timestamp'] + [c for c in SimulationData.columns if c != 'timestamp']
    SimulationData = SimulationData[cols]
    print("Done.")

if(res == 1):
    err_path = OUT_DIR / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            # Print the last 4000 characters to catch the fatal errors at the end
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

EnergyPlus state has been reset.
Starting EnergyPlus Uncontrolled Simulation...
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.
[OCC] Note: no People object matched zone 'PLENUM-1' (will be ignored).
[occ-counter] Resolved 5 People handles across 5 zones.
[co2-outdoor] set=420.0 ppm  read-back=420.0 ppm
Zone :SPACE5-1 Time :2026-01-01 00:10:00

--- [SPACE5-1] Param Extraction ---
Physical: V=447.682556152, M_air=539.01
R_ground: 0.0012
R_env (External Merged): inf
Adjacent Zones Array: [{'zone': 'PLENUM-1', 'R_env': 0.0044, 'handle_T_in': 23}, {'zone': 'SPACE1-1', 'R_env': 0.0045, 'handle_T_in': 25}, {'zone': 'SPACE2-1', 'R_env': 0.0132, 'handle_T_in': 27}, {'zone': 'SPACE3-1', 'R_env': 0.0045, 'handle_T_in': 29}, {'zone': 'SPACE4-1', 'R_env': 0.0132, 'handle_T_in': 31}]
-----------------------------------

[SPACE5-1] Handles initialized. Neighbors: ['PLENUM-1', 'SPACE1-1', 'SPACE2-1', 'S

In [ ]:
# @title plot_zone_results
def plot_zone_results(df):
    """
    Plots a 4-row subplot comparing actual vs predicted state variables.
    Expects a DataFrame with 'timestamp' as the index.
    """

    # Create subplots
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(
            "Zone Air Temperature (T_in)",
            "Thermal Mass Temperature (T_m)",
            "Humidity Ratio (W_in)",
            "CO2 Concentration (C_in)"
        )
    )

    # Color palette based on your design
    colors = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd', '#00d2d3']

    # 1. T_in Plot
    fig.add_trace(go.Scatter(x=df.index, y=df['T_in_actual'], name='Actual T_in', line=dict(color=colors[0])), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['T_in_pred'], name='Predicted T_in', line=dict(color=colors[0], dash='dash')), row=1, col=1)

    # 2. T_m Plot
    fig.add_trace(go.Scatter(x=df.index, y=df['T_m_actual'], name='Actual T_m', line=dict(color=colors[1])), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['T_m_pred'], name='Predicted T_m', line=dict(color=colors[1], dash='dash')), row=2, col=1)

    # 3. W_in Plot
    fig.add_trace(go.Scatter(x=df.index, y=df['W_in_actual'], name='Actual W_in', line=dict(color=colors[2])), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['W_in_pred'], name='Predicted W_in', line=dict(color=colors[2], dash='dash')), row=3, col=1)

    # 4. C_in Plot
    fig.add_trace(go.Scatter(x=df.index, y=df['C_in_actual'], name='Actual C_in', line=dict(color=colors[3])), row=4, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['C_in_pred'], name='Predicted C_in', line=dict(color=colors[3], dash='dash')), row=4, col=1)

    # Layout Updates
    fig.update_layout(
        height=1000,
        title_text="RC Model Prediction vs EnergyPlus Physics Engine Validation",
        hovermode="x unified",
        template="plotly_dark",
        showlegend=True
    )

    # Labels
    fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", row=2, col=1)
    fig.update_yaxes(title_text="Humidity (kg/kg)", row=3, col=1)
    fig.update_yaxes(title_text="CO2 (ppm)", row=4, col=1)
    fig.update_xaxes(title_text="Time", row=4, col=1)

    fig.show()


In [ ]:
# @title simulation_errors_summary

def simulation_errors_summary(df):
    """
    Takes the EnergyPlus vs RC Model DataFrame and calculates
    the numerical differences (errors) between actual and predicted states.
    """
    print("=" * 60)
    print(f"{'MODEL VALIDATION ERROR SUMMARY':^60}")
    print("=" * 60)

    # 1. Define the pairs we want to analyze
    variables = {
        'T_in': {'actual': 'T_in_actual', 'pred': 'T_in_pred', 'unit': '°C', 'name': 'Zone Air Temp'},
        'T_m':  {'actual': 'T_m_actual',  'pred': 'T_m_pred',  'unit': '°C', 'name': 'Thermal Mass Temp'},
        'W_in': {'actual': 'W_in_actual', 'pred': 'W_in_pred', 'unit': 'kg/kg', 'name': 'Humidity Ratio'},
        'C_in': {'actual': 'C_in_actual', 'pred': 'C_in_pred', 'unit': 'ppm', 'name': 'CO2 Concentration'}
    }

    results = []

    # 2. Calculate metrics for each variable
    for var_key, v in variables.items():
        # Check if the columns actually exist in the dataframe before calculating
        if v['actual'] in df.columns and v['pred'] in df.columns:

            # Drop NaN values (like the very first timestep where pred is None)
            valid_data = df[[v['actual'], v['pred']]].dropna()

            if len(valid_data) == 0:
                continue

            # Calculate the raw error (Actual - Predicted)
            error = valid_data[v['actual']] - valid_data[v['pred']]

            # Calculate Statistics
            mae = error.abs().mean()
            max_err = error.abs().max()
            rmse = np.sqrt((error**2).mean())

            # Grab the error at the very last simulated timestep
            final_err = error.iloc[-1]

            results.append({
                'Variable': f"{v['name']} ({var_key})",
                'Unit': v['unit'],
                'MAE (Avg Error)': f"{mae:.4f}",
                'Max Error': f"{max_err:.4f}",
                'RMSE': f"{rmse:.4f}",
                'Final Step Error': f"{final_err:.4f}"
            })

    # 3. Format and print the results
    if not results:
        print("No prediction columns found to compare!")
        return

    summary_df = pd.DataFrame(results)

    # Print the table nicely without the index
    print(summary_df.to_string(index=False))
    print("=" * 60)

    # Return the dataframe in case you want to save it to a CSV later
    return summary_df

In [ ]:
log_data = sim.zones["SPACE5-1"].log

# Convert to DataFrame
df = pd.DataFrame(log_data)
df.set_index("timestamp", inplace=True)

# Remove First Hour
df.index = pd.to_datetime(df.index)
start_time = df.index.min()
cutoff_time = start_time + pd.Timedelta(hours=1)
df_cleaned = df[df.index >= cutoff_time]

simulation_errors_summary(df_cleaned)
plot_zone_results(df)

               MODEL VALIDATION ERROR SUMMARY               
                Variable  Unit MAE (Avg Error) Max Error    RMSE Final Step Error
    Zone Air Temp (T_in)    °C          0.2233    0.5813  0.2555          -0.1841
 Thermal Mass Temp (T_m)    °C          0.0447    0.2645  0.0577          -0.0424
   Humidity Ratio (W_in) kg/kg          0.0001    0.0005  0.0001          -0.0001
CO2 Concentration (C_in)   ppm         11.6557  102.7968 14.5291          19.4344
